# Comprehensive Dataset Comparison

This notebook provides a comprehensive comparison of three datasets:

- **Original (Org)**: Original training dataset
- **Generated (Gen)**: Synthetic dataset generated via LLM
- **Paraphrased (Para)**: Synthetic dataset via paraphrasing

**Metrics compared:**
1. Lexical diversity (TTR, vocabulary richness)
2. Text characteristics (sentence length statistics)
3. Emotion distribution analysis
4. Statistical comparisons between distributions

In [ ]:
%pip install -q pandas transformers scikit-learn

In [ ]:
import pandas as pd
from typing import List, Set, Dict
from transformers import AutoTokenizer
from collections import Counter
import numpy as np
from IPython.display import display

# Load CafeBERT tokenizer
tokenizer = AutoTokenizer.from_pretrained("uitnlp/CafeBERT")

## Analysis Functions

In [ ]:
def get_tokens(texts: List[str]) -> List[str]:
    """Tokenize texts using CafeBERT tokenizer."""
    tokens = []
    for text in texts:
        if pd.isna(text):
            continue
        words = tokenizer.tokenize(str(text).lower())
        tokens.extend(words)
    return tokens

def get_types(tokens: List[str]) -> Set[str]:
    """Get unique word types from tokens."""
    return set(tokens)

def compute_ttr(texts: List[str]) -> dict:
    """Compute Type-Token Ratio (TTR)."""
    tokens = get_tokens(texts)
    types = get_types(tokens)
    
    num_tokens = len(tokens)
    num_types = len(types)
    ttr = num_types / num_tokens if num_tokens > 0 else 0
    
    return {
        'types': num_types,
        'tokens': num_tokens,
        'ttr': ttr
    }

def compute_jaccard(texts_a: List[str], texts_b: List[str]) -> dict:
    """Compute Jaccard Index between two datasets."""
    types_a = get_types(get_tokens(texts_a))
    types_b = get_types(get_tokens(texts_b))
    
    intersection = types_a & types_b
    union = types_a | types_b
    
    jaccard = len(intersection) / len(union) if len(union) > 0 else 0
    
    return {
        'types_a': len(types_a),
        'types_b': len(types_b),
        'intersection': len(intersection),
        'union': len(union),
        'jaccard': jaccard
    }

def compute_sentence_stats(texts: List[str]) -> dict:
    """Compute sentence length statistics."""
    lengths = [len(str(text)) for text in texts if pd.notna(text)]
    if not lengths:
        return {'mean': 0, 'median': 0, 'min': 0, 'max': 0, 'std': 0}

    mean_val = sum(lengths) / len(lengths)
    sorted_lengths = sorted(lengths)
    median_val = sorted_lengths[len(lengths)//2]

    variance = sum((x - mean_val)**2 for x in lengths) / len(lengths)
    std_val = variance**0.5

    return {
        'mean': mean_val,
        'median': median_val,
        'min': min(lengths),
        'max': max(lengths),
        'std': std_val
    }

def compute_vocab_richness(texts: List[str]) -> dict:
    """Compute vocabulary richness metrics."""
    tokens = get_tokens(texts)
    types = get_types(tokens)

    if not tokens:
        return {'hapax_ratio': 0, 'hapax_count': 0, 'low_freq_ratio': 0, 'high_freq_coverage': 0, 'vocab_size': 0, 'total_tokens': 0}

    freq_dist = Counter(tokens)

    # Hapax legomena (words appearing exactly once)
    hapax = sum(1 for count in freq_dist.values() if count == 1)

    # Words appearing 2-10 times
    low_freq = sum(1 for count in freq_dist.values() if 2 <= count <= 10)

    # High frequency words (top 10%)
    sorted_freqs = sorted(freq_dist.values(), reverse=True)
    top_10_percent = int(len(sorted_freqs) * 0.1)
    high_freq_sum = sum(sorted_freqs[:top_10_percent]) if sorted_freqs else 0

    return {
        'hapax_ratio': hapax / len(types) if types else 0,
        'hapax_count': hapax,
        'low_freq_ratio': low_freq / len(types) if types else 0,
        'high_freq_coverage': high_freq_sum / len(tokens) if tokens else 0,
        'vocab_size': len(types),
        'total_tokens': len(tokens)
    }

def analyze_emotion_distribution(df: pd.DataFrame, emotion_col: str = 'Emotion') -> dict:
    """Analyze emotion distribution in dataset."""
    if emotion_col not in df.columns:
        return {'error': f'Column {emotion_col} not found'}

    emotion_counts = df[emotion_col].value_counts()
    total = len(df)

    # Calculate entropy
    entropy = 0
    for count in emotion_counts:
        if count > 0:
            prob = count / total
            entropy -= prob * np.log2(prob)

    return {
        'distribution': emotion_counts.to_dict(),
        'total_samples': total,
        'num_classes': len(emotion_counts),
        'entropy': entropy,
        'most_common': emotion_counts.index[0] if len(emotion_counts) > 0 else None,
        'least_common': emotion_counts.index[-1] if len(emotion_counts) > 0 else None
    }

def compute_statistical_comparison(dist1: dict, dist2: dict) -> dict:
    """Compute statistical comparison between two emotion distributions."""
    if 'error' in dist1 or 'error' in dist2:
        return {'error': 'Invalid distributions'}

    dist1_counts = dist1['distribution']
    dist2_counts = dist2['distribution']

    all_emotions = set(dist1_counts.keys()) | set(dist2_counts.keys())

    # Chi-square test
    chi_square = 0
    total1 = dist1['total_samples']
    total2 = dist2['total_samples']
    total_both = total1 + total2

    for emotion in all_emotions:
        count1 = dist1_counts.get(emotion, 0)
        count2 = dist2_counts.get(emotion, 0)
        total_emotion = count1 + count2

        expected1 = total_emotion * total1 / total_both if total_both > 0 else 0
        expected2 = total_emotion * total2 / total_both if total_both > 0 else 0

        if expected1 > 0:
            chi_square += (count1 - expected1) ** 2 / expected1
        if expected2 > 0:
            chi_square += (count2 - expected2) ** 2 / expected2

    # KL divergence
    kl_div = 0
    for emotion in all_emotions:
        p = dist1_counts.get(emotion, 0) / total1 if total1 > 0 else 0
        q = dist2_counts.get(emotion, 0) / total2 if total2 > 0 else 0

        if p > 0 and q > 0:
            kl_div += p * np.log2(p / q)

    # Distribution overlap
    overlap = len(set(dist1_counts.keys()) & set(dist2_counts.keys())) / len(all_emotions) if all_emotions else 0

    return {
        'chi_square_statistic': chi_square,
        'kl_divergence': kl_div,
        'distribution_overlap': overlap,
        'shared_emotions': len(set(dist1_counts.keys()) & set(dist2_counts.keys())),
        'unique_to_first': len(set(dist1_counts.keys()) - set(dist2_counts.keys())),
        'unique_to_second': len(set(dist2_counts.keys()) - set(dist1_counts.keys()))
    }

## Load Data

In [ ]:
# Load three datasets
import os

base_path = 'data/processed'
org_path = os.path.join(base_path, 'train_org_processed.csv')
gen_path = os.path.join(base_path, 'train_1071_gen_processed.csv')
para_path = os.path.join(base_path, 'train_1071_para_processed.csv')

org_df = pd.read_csv(org_path)
gen_df = pd.read_csv(gen_path)
para_df = pd.read_csv(para_path)

print(f"Original dataset: {len(org_df)} samples")
print(f"Generated dataset: {len(gen_df)} samples")
print(f"Paraphrased dataset: {len(para_df)} samples")
print(f"\nColumns in datasets:")
print(f"  Org: {list(org_df.columns)}")
print(f"  Gen: {list(gen_df.columns)}")
print(f"  Para: {list(para_df.columns)}")

## Extract Texts and Compute Basic Metrics

In [ ]:
# Extract texts
text_column = 'Sentence' if 'Sentence' in org_df.columns else 'Sentence_clean'
org_texts = org_df[text_column].tolist()
gen_texts = gen_df[text_column].tolist()
para_texts = para_df[text_column].tolist()

# Compute TTR for all three datasets
print("Computing TTR for all datasets...")
ttr_org = compute_ttr(org_texts)
ttr_gen = compute_ttr(gen_texts)
ttr_para = compute_ttr(para_texts)

print("\n=== Type-Token Ratio (TTR) ===")
print(f"\nOriginal Dataset:")
print(f"  Types (unique words): {ttr_org['types']:,}")
print(f"  Tokens (total words): {ttr_org['tokens']:,}")
print(f"  TTR: {ttr_org['ttr']:.4f}")

print(f"\nGenerated Dataset:")
print(f"  Types (unique words): {ttr_gen['types']:,}")
print(f"  Tokens (total words): {ttr_gen['tokens']:,}")
print(f"  TTR: {ttr_gen['ttr']:.4f}")

print(f"\nParaphrased Dataset:")
print(f"  Types (unique words): {ttr_para['types']:,}")
print(f"  Tokens (total words): {ttr_para['tokens']:,}")
print(f"  TTR: {ttr_para['ttr']:.4f}")

## Compute Jaccard Index Comparisons

In [ ]:
# Compute Jaccard Index for all pairwise comparisons
print("Computing Jaccard Index for all pairs...")

jaccard_gen_para = compute_jaccard(gen_texts, para_texts)
jaccard_gen_org = compute_jaccard(gen_texts, org_texts)
jaccard_para_org = compute_jaccard(para_texts, org_texts)

print("\n=== Jaccard Index Comparisons ===")

print(f"\n1. Generated vs Paraphrased:")
print(f"   Gen types: {jaccard_gen_para['types_a']:,}")
print(f"   Para types: {jaccard_gen_para['types_b']:,}")
print(f"   Intersection: {jaccard_gen_para['intersection']:,}")
print(f"   Union: {jaccard_gen_para['union']:,}")
print(f"   Jaccard Index: {jaccard_gen_para['jaccard']:.4f}")

print(f"\n2. Generated vs Original:")
print(f"   Gen types: {jaccard_gen_org['types_a']:,}")
print(f"   Org types: {jaccard_gen_org['types_b']:,}")
print(f"   Intersection: {jaccard_gen_org['intersection']:,}")
print(f"   Union: {jaccard_gen_org['union']:,}")
print(f"   Jaccard Index: {jaccard_gen_org['jaccard']:.4f}")

print(f"\n3. Paraphrased vs Original:")
print(f"   Para types: {jaccard_para_org['types_a']:,}")
print(f"   Org types: {jaccard_para_org['types_b']:,}")
print(f"   Intersection: {jaccard_para_org['intersection']:,}")
print(f"   Union: {jaccard_para_org['union']:,}")
print(f"   Jaccard Index: {jaccard_para_org['jaccard']:.4f}")

## Comprehensive Dataset Comparison

In [ ]:
# Compute comprehensive metrics for all datasets
print("Computing comprehensive metrics...")

# Sentence statistics
print("\n=== Sentence Length Statistics ===")
sent_stats_org = compute_sentence_stats(org_texts)
sent_stats_gen = compute_sentence_stats(gen_texts)
sent_stats_para = compute_sentence_stats(para_texts)

sent_stats_df = pd.DataFrame({
    'Dataset': ['Original', 'Generated', 'Paraphrased'],
    'Mean': [f"{sent_stats_org['mean']:.1f}", f"{sent_stats_gen['mean']:.1f}", f"{sent_stats_para['mean']:.1f}"],
    'Median': [sent_stats_org['median'], sent_stats_gen['median'], sent_stats_para['median']],
    'Min': [sent_stats_org['min'], sent_stats_gen['min'], sent_stats_para['min']],
    'Max': [sent_stats_org['max'], sent_stats_gen['max'], sent_stats_para['max']],
    'Std': [f"{sent_stats_org['std']:.1f}", f"{sent_stats_gen['std']:.1f}", f"{sent_stats_para['std']:.1f}"]
})
print(sent_stats_df.to_string(index=False))

# Vocabulary richness
print("\n=== Vocabulary Richness Metrics ===")
vocab_org = compute_vocab_richness(org_texts)
vocab_gen = compute_vocab_richness(gen_texts)
vocab_para = compute_vocab_richness(para_texts)

vocab_df = pd.DataFrame({
    'Dataset': ['Original', 'Generated', 'Paraphrased'],
    'Vocab Size': [vocab_org['vocab_size'], vocab_gen['vocab_size'], vocab_para['vocab_size']],
    'Hapax Ratio': [f"{vocab_org['hapax_ratio']:.3f}", f"{vocab_gen['hapax_ratio']:.3f}", f"{vocab_para['hapax_ratio']:.3f}"],
    'Low Freq Ratio': [f"{vocab_org['low_freq_ratio']:.3f}", f"{vocab_gen['low_freq_ratio']:.3f}", f"{vocab_para['low_freq_ratio']:.3f}"],
    'High Freq Coverage': [f"{vocab_org['high_freq_coverage']:.3f}", f"{vocab_gen['high_freq_coverage']:.3f}", f"{vocab_para['high_freq_coverage']:.3f}"]
})
print(vocab_df.to_string(index=False))

## Emotion Distribution Analysis

In [ ]:
# Emotion distribution analysis
print("\n=== Emotion Distribution Analysis ===")

emo_org = analyze_emotion_distribution(org_df)
emo_gen = analyze_emotion_distribution(gen_df)
emo_para = analyze_emotion_distribution(para_df)

print(f"\nOriginal Dataset:")
print(f"  Total samples: {emo_org['total_samples']}")
print(f"  Number of classes: {emo_org['num_classes']}")
print(f"  Most common emotion: {emo_org['most_common']}")
print(f"  Distribution entropy: {emo_org['entropy']:.3f}")

print(f"\nGenerated Dataset:")
print(f"  Total samples: {emo_gen['total_samples']}")
print(f"  Number of classes: {emo_gen['num_classes']}")
print(f"  Most common emotion: {emo_gen['most_common']}")
print(f"  Distribution entropy: {emo_gen['entropy']:.3f}")

print(f"\nParaphrased Dataset:")
print(f"  Total samples: {emo_para['total_samples']}")
print(f"  Number of classes: {emo_para['num_classes']}")
print(f"  Most common emotion: {emo_para['most_common']}")
print(f"  Distribution entropy: {emo_para['entropy']:.3f}")

# Statistical comparisons
print(f"\n=== Statistical Comparisons Between Emotion Distributions ===")

comp_gen_org = compute_statistical_comparison(emo_gen, emo_org)
comp_para_org = compute_statistical_comparison(emo_para, emo_org)
comp_gen_para = compute_statistical_comparison(emo_gen, emo_para)

print(f"\nGenerated vs Original:")
print(f"  Chi-square statistic: {comp_gen_org['chi_square_statistic']:.2f}")
print(f"  KL divergence: {comp_gen_org['kl_divergence']:.4f}")
print(f"  Distribution overlap: {comp_gen_org['distribution_overlap']:.3f}")
print(f"  Shared emotions: {comp_gen_org['shared_emotions']}")

print(f"\nParaphrased vs Original:")
print(f"  Chi-square statistic: {comp_para_org['chi_square_statistic']:.2f}")
print(f"  KL divergence: {comp_para_org['kl_divergence']:.4f}")
print(f"  Distribution overlap: {comp_para_org['distribution_overlap']:.3f}")
print(f"  Shared emotions: {comp_para_org['shared_emotions']}")

print(f"\nGenerated vs Paraphrased:")
print(f"  Chi-square statistic: {comp_gen_para['chi_square_statistic']:.2f}")
print(f"  KL divergence: {comp_gen_para['kl_divergence']:.4f}")
print(f"  Distribution overlap: {comp_gen_para['distribution_overlap']:.3f}")
print(f"  Shared emotions: {comp_gen_para['shared_emotions']}")

## Final Comprehensive Summary

In [ ]:
# Create comprehensive summary table
print("\n=== COMPREHENSIVE DATASET COMPARISON SUMMARY ===")

summary_data = {
    'Metric': [
        'Sample Size', 'Vocabulary Size', 'TTR', 'Sentence Length (Mean)',
        'Hapax Ratio', 'Emotion Classes', 'Distribution Entropy',
        'Jaccard vs Org', 'Jaccard vs Gen', 'Jaccard vs Para'
    ],
    'Original': [
        len(org_df),
        ttr_org['types'],
        f"{ttr_org['ttr']:.4f}",
        f"{sent_stats_org['mean']:.1f}",
        f"{vocab_org['hapax_ratio']:.3f}",
        emo_org['num_classes'],
        f"{emo_org['entropy']:.3f}",
        '-', '-', '-'
    ],
    'Generated': [
        len(gen_df),
        ttr_gen['types'],
        f"{ttr_gen['ttr']:.4f}",
        f"{sent_stats_gen['mean']:.1f}",
        f"{vocab_gen['hapax_ratio']:.3f}",
        emo_gen['num_classes'],
        f"{emo_gen['entropy']:.3f}",
        f"{jaccard_gen_org['jaccard']:.4f}",
        '-', 
        f"{jaccard_gen_para['jaccard']:.4f}"
    ],
    'Paraphrased': [
        len(para_df),
        ttr_para['types'],
        f"{ttr_para['ttr']:.4f}",
        f"{sent_stats_para['mean']:.1f}",
        f"{vocab_para['hapax_ratio']:.3f}",
        emo_para['num_classes'],
        f"{emo_para['entropy']:.3f}",
        f"{jaccard_para_org['jaccard']:.4f}",
        f"{jaccard_gen_para['jaccard']:.4f}",
        '-'
    ]
}

comprehensive_summary = pd.DataFrame(summary_data)
print(comprehensive_summary.to_string(index=False))

# Display the table
display(comprehensive_summary)

print(f"\n=== KEY FINDINGS ===")
print(f"1. Sample sizes: Org={len(org_df)}, Gen={len(gen_df)}, Para={len(para_df)}")
print(f"2. Vocabulary sizes: Org={ttr_org['types']}, Gen={ttr_gen['types']}, Para={ttr_para['types']}")
print(f"3. TTR values: Org={ttr_org['ttr']:.4f}, Gen={ttr_gen['ttr']:.4f}, Para={ttr_para['ttr']:.4f}")
print(f"4. Sentence lengths: Org={sent_stats_org['mean']:.1f}, Gen={sent_stats_gen['mean']:.1f}, Para={sent_stats_para['mean']:.1f}")
print(f"5. Vocabulary overlap (Gen vs Org): {jaccard_gen_org['jaccard']:.4f}")
print(f"6. Vocabulary overlap (Para vs Org): {jaccard_para_org['jaccard']:.4f}")
print(f"7. Vocabulary overlap (Gen vs Para): {jaccard_gen_para['jaccard']:.4f}")
print(f"8. Emotion distribution similarities vary significantly between datasets")
print(f"9. All datasets maintain the same emotion classes but with different distributions")